In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier

In [ ]:
file_path = "../data/raw/churn-bigml-80.csv"
df = pd.read_csv(file_path)

print(f"Dataset shape: {df.shape}")
print("\nChurn distribution:")
print(df["Churn"].value_counts())
df.head()

In [ ]:
# Fix data types
df["State"] = df["State"].astype("category")
df["Area code"] = df["Area code"].astype("category")
df["International plan"] = df["International plan"].map({"Yes": 1, "No": 0})
df["Voice mail plan"] = df["Voice mail plan"].map({"Yes": 1, "No": 0})

# Feature engineering
df_fe = df.copy()
df_fe = df_fe.drop(columns=["State", "Area code"])

df_fe["total_minutes"] = (
    df_fe["Total day minutes"] +
    df_fe["Total eve minutes"] +
    df_fe["Total night minutes"] +
    df_fe["Total intl minutes"]
)

df_fe["total_charges"] = (
    df_fe["Total day charge"] +
    df_fe["Total eve charge"] +
    df_fe["Total night charge"] +
    df_fe["Total intl charge"]
)

df_fe["high_service_calls"] = (df_fe["Customer service calls"] >= 3).astype(int)
df_fe["intl_usage_ratio"] = (df_fe["Total intl minutes"] / df_fe["total_minutes"]).fillna(0)

df_fe["account_length_group"] = pd.cut(
    df_fe["Account length"],
    bins=[0, 50, 150, df_fe["Account length"].max()],
    labels=["new", "mid", "long"]
)

categorical_cols = ["International plan", "Voice mail plan", "account_length_group"]
df_fe = pd.get_dummies(df_fe, columns=categorical_cols, drop_first=True)
df_fe = df_fe.fillna(0)  # Handle any NaNs from pd.cut edge cases

# Scale numeric features
numeric_cols = df_fe.select_dtypes(include=["int64", "float64"]).columns
numeric_cols = numeric_cols.drop("Churn", errors="ignore")

scaler = StandardScaler()
df_fe[numeric_cols] = scaler.fit_transform(df_fe[numeric_cols])

X = df_fe.drop("Churn", axis=1)
y = df_fe["Churn"].astype(int)

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

In [ ]:
# Build and train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)

# Evaluate
print("Random Forest Results")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"F1 Score (Churn): {f1_score(y_test, y_pred_rf):.4f}")
print()
print(classification_report(y_test, y_pred_rf, target_names=["No Churn", "Churn"]))

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(
    confusion_matrix(y_test, y_pred_rf),
    annot=True, fmt="d", cmap="Greens",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)
plt.title("Random Forest - Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()

In [ ]:
feat_importance = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

feat_importance.head(10).plot(kind="bar", figsize=(10, 5), color="forestgreen", edgecolor="black")
plt.title("Random Forest - Top 10 Most Important Features")
plt.ylabel("Importance Score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("\nTop 10 Features:")
print(feat_importance.head(10).round(4))